# $\log(4/3)$: historical CMF bound

Each section constructs a CMF, walks its trajectory, and prints $\delta_N$ at source depth 1000 for the reduced rational approximant. A finite-depth value does not prove the published measure bound.

In [ ]:
# @title Install dependencies (run once per environment)
%pip -q install ramanujantools==0.0.8 sympy==1.14.0 mpmath==1.3.0 matplotlib==3.11.2

## Setup

Standard hypergeometric fields come from ramanujantools; the four-axis Huttner moment field is constructed in `cmfs.py`.


In [ ]:
import sys

import matplotlib.pyplot as plt
import sympy as sp
from flint import arb, ctx

from cmfs import huttner_logs, huttner_logs_seeds
from ramanujantools import Matrix
from ramanujantools.cmf.pfq import pFq

if hasattr(sys, "set_int_max_str_digits"):
    sys.set_int_max_str_digits(0)
with ctx.workdps(7000):
    TARGET = arb(4).log() - arb(3).log()
DEPTHS = list(range(10, 1001, 10))
x0, x1, y0 = sp.symbols("x0 x1 y0")
results = {}


def delta(approximant):
    """Finite-depth exponent of the reduced rational approximant."""
    p, q = int(approximant.p), int(approximant.q)
    with ctx.workprec(2*q.bit_length() + 128):
        error = abs(TARGET - arb(p)/q)
        assert error > 0
        return float((-1 - error.log()/arb(q).log()).mid())

## 1. Alladi–Robinson (1980): $\log(4/3)$

[Integral construction and measure formula](https://www.math.ru.nl/~wzudilin/gat/oALLADI4.txt).
$$E_n=\int_0^1\frac{t^n(1-t)^n}{(t+3)^{n+1}}\,dt.$$
**CMF:** rank 2, `${}_2F_1(-1/3)` ? **Start:** $(2,2;4)$ ? **Direction:** $(1,1;2)$.
The readout is $E_n=C_n+B_n\log(4/3)$, $p_n/q_n=-C_n/B_n$. Depth $N$ means the integral index $n=N$.
The beta-integral normalization is common to both coefficient rows and cancels in the readout; the code uses the $(F,\theta F)$ basis.


In [ ]:
cmf = pFq(2, 1, -sp.Rational(1, 3))
start = {x0: 2, x1: 2, y0: 4}
trajectory = {x0: 1, x1: 1, y0: 2}
walks = cmf.walk(trajectory, [N - 1 for N in DEPTHS], start)
C = Matrix([[-8, 23]])
B = Matrix([[28, -80]])
approximants = [-(C*M)[0]/(B*M)[0] for M in walks]

deltas = [delta(r) for r in approximants]
results["Alladi–Robinson (1980)"] = (DEPTHS, deltas)
print(f"delta at depth 1000: {deltas[-1]:.12f}")

## 2. Huttner (2003): simultaneous logarithmic construction

[Corollary 3.1(4), pp. 146–147](https://www.jstage.jst.go.jp/article/kyushujm/57/1/57_1_129/_pdf)
uses Hata's method to bound integer linear forms in $1,\log2,\log3$ with
exponent $10.1017577\ldots$. Consequently every nonzero rational combination
of the two logarithms has $\mu\le11.1017577\ldots$, including this target.
The measure comparison scale is $\delta_{\rm eq}=1/10.1017577\ldots\approx0.09899267$.
This simultaneous bound is weaker than the individual Alladi–Robinson bound;
it comes from a distinct approximation sequence.

The paper's remainders are
$$R_{i,n}=\int_{I_i}\frac{[t(t-\tfrac12)(t-\tfrac13)]^{2n}}{(t-1)^{3n}}\,dt,
\qquad I_1=[0,\tfrac13],\quad I_2=[\tfrac13,\tfrac12].$$
For this target use $I_i=[1/3,1/2]$. Substitution $z=1-t$ gives, up to the
common sign $(-1)^{n+1}$, the moment weight
$$(z-1)^u(z-\tfrac23)^v(z-\tfrac12)^w z^{-m}.$$
The full field has **rank 3 and four axes** $(u,v,w,m)$, with start
$(2,2,2,3)$ and direction $(2,2,2,3)$. The source index is $n=k+1$ for a
walk of $k$ steps. The moments $j=0,1,2$ at $n=1$ give rows $P,Q$ by exact
Laurent integration, with $J_j=P_j-Q_j\log(4/3)$; the readout is
$$(p_n/q_n)=\frac{(PW_{n-1})_0}{(QW_{n-1})_0}.$$

The paper gives scaled remainder growth factors approximately $0.34171$
and $0.10387$, and coefficient growth factor $51322.26$. Thus the two
remainder exponents differ: the faster second interval has a comparison
exponent about $0.20880$, although the shared measure bound uses the slower
first interval. Reduced denominators and finite depth can further change
the measured $\delta_n$.

Validation: all three transported coefficient pairs agree exactly with
direct integration at source indices 1–7; all six axis pairs pass at two
regular points; $\delta_{1000}$ is checked with overlapping Arb intervals
at two precisions. Local verification files are kept in ignored `local_checks/`.


In [7]:
u, v, w, m = sp.symbols("u v w m")
cmf = huttner_logs()
start = {u: 2, v: 2, w: 2, m: 3}
trajectory = {u: 2, v: 2, w: 2, m: 3}
P, Q = huttner_logs_seeds(sp.Rational(4, 3))
walks = cmf.walk(trajectory, [N - 1 for N in DEPTHS], start)
approximants = [(P * W)[0] / (Q * W)[0] for W in walks]
deltas = [delta(r) for r in approximants]
results["Huttner (2003), simultaneous"] = (DEPTHS, deltas)
print(f"delta at source index 1000: {deltas[-1]:.15f}")


delta at source index 1000: 0.209189334972855


## All constructions

$\delta_N$ against the source depth $N$.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
for name, (depths, values) in results.items():
    ax.plot(depths, values, label=name)
ax.set(xlabel="$N$", ylabel=r"$\delta_N$")
ax.legend(frameon=False)
plt.show()

## Historical overview

| Construction | Published upper bound $\mu_0$ | $\delta_{\rm eq}$ | $\delta_{1000}$ | CMF |
|:--|--:|--:|--:|:--|
| Alladi–Robinson | 3.224053288 | 0.449629514425 | 0.460862050522 | rank-2 ${}_2F_1$ walk |
| Huttner (2003), simultaneous | $11.1017577\ldots$ | $0.09899267$ | $0.209189334973$ | rank 3, four axes |

$\delta_{\rm eq}=1/(\mu_0-1)$ is a comparison scale, not the finite-depth exponent.
